In [38]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from openai.lib._pydantic import to_strict_json_schema

from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json

from textwrap import dedent

from pathlib import Path


from time import perf_counter

# Preliminari

In [ ]:
# Configurazione OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
ANN_MODEL = constants.RectalCancerStagingData
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0
MODEL = constants.OPENAI_GPT_4_1_MINI
UPLOAD_DATA = False

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, ANN_MODEL)

In [41]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [42]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

train_data, validation_data = data['train'], data['validation']

################################
# Convert float columns to Int64
################################
float_cols = train_data.select_dtypes("float").columns
for col in float_cols:
    train_data[col] = train_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(train_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(validation_data) = }")

len(train_data) = 184
len(validation_data) = 66


# Upload data to Openai dashboard

## Create local fine-tuning datasets

Il dataset jsonl dovrà contenere righe con il formato chat completetion. Ogni riga dovrà essere:

```
{
  "messages": [
    { "role": "system", "content": <system_prompt>}
    { "role": "user", "content": <report_text> },
    { "role": "assistant", "content": <correct_annotation> },
  ]
}
```

In [ ]:
def prepare_example_conversation(row):
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row['report_text']},
            {"role": "assistant", "content": from_series_to_basemodel(row, ANN_MODEL).model_dump_json()},
        ]
    }
    
def write_jsonl(data_list: list, filename: str) -> None:
    with open(filename, "w") as out:
        for ddict in data_list:
            jout = json.dumps(ddict) + "\n"
            out.write(jout)
            
def upload_file(file_name: str, purpose: str) -> str:
    with open(file_name, "rb") as file_fd:
        response = client.files.create(file=file_fd, purpose=purpose)
    return response.id

In [ ]:
if UPLOAD_DATA:
    train_list = train_data.apply(prepare_example_conversation, axis=1).tolist()
    validation_list = validation_data.apply(prepare_example_conversation, axis=1).tolist()
    
    training_file_path = base_dir / 'data' / 'ft-dataset' / constants.OPENAI_TRAIN_FILE_NAME
    write_jsonl(train_list, training_file_path)

    validation_file_path = base_dir / 'data' / 'ft-dataset' / constants.OPENAI_VALIDATION_FILE_NAME
    write_jsonl(validation_list, validation_file_path)
    
    training_file_id = upload_file(training_file_path, "fine-tune")
    validation_file_id = upload_file(validation_file_path, "fine-tune")

    print("Training file ID:", training_file_id)
    print("Validation file ID:", validation_file_id)

# Fine tuning

In [ ]:
print(MODEL)

'gpt-4.1-mini-2025-04-14'

In [44]:
fine_tuning_job = client.fine_tuning.jobs.create(
  training_file=training_file_id,
  validation_file=validation_file_id,
  model=MODEL,
  seed=constants.SEED,
  suffix="report-extractor"
)

job_id = fine_tuning_job.id

In [51]:
print("Job ID:", fine_tuning_job.id)
print("Status:", fine_tuning_job.status)

Job ID: ftjob-a0jD5zL6mx2u3OhXi2eJC1IL
Status: validating_files


In [52]:
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-a0jD5zL6mx2u3OhXi2eJC1IL
Status: running
Trained Tokens: None


In [53]:
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 157/552: training loss=0.02
Step 158/552: training loss=0.04
Step 159/552: training loss=0.05
Step 160/552: training loss=0.04, validation loss=0.09
Step 161/552: training loss=0.02
Step 162/552: training loss=0.02
Step 163/552: training loss=0.02
Step 164/552: training loss=0.00
Step 165/552: training loss=0.01
Step 166/552: training loss=0.03
Step 167/552: training loss=0.03
Step 168/552: training loss=0.04
Step 169/552: training loss=0.02
Step 170/552: training loss=0.07, validation loss=0.00
Step 171/552: training loss=0.01
Step 172/552: training loss=0.08
Step 173/552: training loss=0.02
Step 174/552: training loss=0.02
Step 175/552: training loss=0.04
Step 176/552: training loss=0.02


In [50]:
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model

if fine_tuned_model_id is None:
    raise RuntimeError(
        "Fine-tuned model ID not found. Your job has likely not been completed yet."
    )

print("Fine-tuned model ID:", fine_tuned_model_id)

RuntimeError: Fine-tuned model ID not found. Your job has likely not been completed yet.